# 0. Setting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/github-main/

In [ ]:
! pip install torch-geometric --quiet

# 1. Data Generation

In [ ]:
# Baseline Dataset Generation
! python src/data/multiplex_generator_v3.py \
  --size 1500 \
  --seed 1024 \
  --out_dir data/multiplex_baseline \
  --config configs/generator_baseline.json

  # Baseline Data Transformation
! python src/data/build_pyg_dataset_v3.py \
  --manifest data/multiplex_baseline/multiplex.json \
  --out_path data/multiplex_baseline/pyg_data.pt

In [ ]:
# Hard Dataset Generation
! python src/data/multiplex_generator_v3.py \
  --size 1500 \
  --seed 1024 \
  --out_dir data/multiplex_hard \
  --config configs/generator_hard.json

# Data Transformation
! python src/data/build_pyg_dataset_v3.py \
  --manifest data/multiplex_hard/multiplex.json \
  --out_path data/multiplex_hard/pyg_data.pt

# 2. Data Statistics


In [ ]:
!python src/data/basic_diagnostics_v3.py \
  --manifest data/multiplex_baseline/multiplex.json \
  --out_dir data/analysis/multiplex_baseline

# 3. GNN Models Solution

## 3-1. High-Value Target(HVT) Classification

In [ ]:
# Baseline
! python src/models/train_hvt_gnn_v3.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --pos_weight 10 \
  --edge_attr_agg --edge_attr_transform none \
  --include_edge_flags

# Hard
! python src/models/train_hvt_gnn_v3.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --pos_weight 10 \
  --edge_attr_agg --edge_attr_transform none \
  --include_edge_flags

In [ ]:
# HVT Single Task

# Baseline
! python src/models/train_hvt_gnn_v2.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.5 \
  --pos_weight 5 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 1024 \
  --pos_weight 10

  # Hard
! python src/models/train_hvt_gnn_v2.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.5 \
  --pos_weight 5 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 1024 \
  --pos_weight 10

## 3-2. Multi-Tasking
 - Role Classification
 - HVT Classification
 - importance score Regression

In [ ]:
# Baseline
! python src/models/train_multitask_gnn_v3.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --encoder transformer \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 2e-3 --weight_decay 1e-4 --epochs 300 \
  --seed 2025 --patience 50 \
  --edge_attr_transform none \
  --include_edge_flags


# Hard
! python src/models/train_multitask_gnn_v3.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --encoder transformer \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 2e-3 --weight_decay 1e-4 --epochs 300 \
  --seed 2025 --patience 50 \
  --edge_attr_transform none \
  --include_edge_flags

## 3-3. Specific node link prediction
 - Finance layer illegal fund link prediction
 - Communication Layer Contact Prediction


In [ ]:
# Finance layer prediction
# Baseline
! python src/models/train_linkpred_layer_v3.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --layer finance \
  --hidden_dim 192 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags


# Hard
! python src/models/train_linkpred_layer_v3.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --layer finance \
  --hidden_dim 192 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags

In [ ]:
# Communication Layer prediction
# Baseline
! python src/models/train_linkpred_layer_v3.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --layer communication \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags


# Hard
! python src/models/train_linkpred_layer_v3.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --layer communication \
  --hidden_dim 128 --num_layers 2 --dropout 0.3 \
  --lr 1e-3 --weight_decay 1e-4 --epochs 500 \
  --seed 1024 --patience 50 --min_delta 1e-3 \
  --edge_attr_agg --include_edge_flags

# 4. Result Visualization

In [ ]:
# Evaluation Summary and Plots
!python src/analysis/plot_multitask_linkpred_summary.py \
  --run_dirs \
    data/multiplex_baseline \
    data/multiplex_hard \
  --out_dir results/summary_all